In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("../data/bridges_witheverything.csv")

# Qualitative scales to quantitative
earthquake_dict = {"I": 3, "II":2, 'III': 1}
flood_dict = {'A':4, 'B':3, 'C':2, 'D':1}
erosion_dict = {'Riverbank Erosion': 1}
condition_dict = {'A':1, 'B':2, 'C':3, 'D':4}

# Data cleaning and formating
df = df.rename(columns = {'ZONE': 'Earthquake risk', 'NEWFIELD1': 'Flood risk', 'AFFECTED':'Erosion risk'})
df = df.sort_values(by = 'id', ascending = True)
df['id'] = df['id'].astype(str).str.strip()

In [9]:
# Checking missing values
print(f"{df['Flood risk'].isna().sum()} bridges have missing flood risk value")
print(f"{df['Earthquake risk'].isna().sum()} bridges have missing earthquake risk value")
print(f"{df['condition'].isna().sum()} bridges have missing condition value")
# missing values of river erosion risk mean there is no river erosion

59 bridges have missing flood risk value
0 bridges have missing earthquake risk value
0 bridges have missing condition value


In [10]:
# Most missing bridges with missing flood risk values have an estimated value of D
df["Flood risk"] = df["Flood risk"].fillna('D')
# Three of these bridges have a flood risk of at least 3, by visual estimations in GIS
df.loc[df['id'] == '1001602', 'Flood risk'] = 'B'
df.loc[df["id"] == "1001609", "Flood risk"] = 'B'
df.loc[df["id"] == "1001614", "Flood risk"] = 'B'

# Changing qualitative scales to quantitative

df['Flood risk eq'] = df["Flood risk"].map(flood_dict)
df['Earthquake risk eq'] = df['Earthquake risk'].map(earthquake_dict)
df['Condition eq'] = df["condition"].map(condition_dict)

# bridges in areas where there is no erosion risk are assigned a 0
df["Erosion risk eq"] = df["Erosion risk"].map(erosion_dict).fillna(0)

df.head(5)

,road,id,model_type,name,lat,lon,length,condition,lrp,Earthquake risk,Flood risk,Erosion risk,Flood risk eq,Earthquake risk eq,Condition eq,Erosion risk eq
18,N1,1000015,bridge,Kachpur bridge,23.703347,90.517305,0,B,LRP008a,II,D,NaN,1,2,2,0.0
19,N1,1000025,bridge,NOYAPARA CULVERT,23.694361,90.537611,6,A,LRP011a,II,D,NaN,1,2,1,0.0
20,N1,1000027,bridge,NAYABARI KASPUR BOX CULVERT,23.692277,90.541055,8,A,LRP011b,II,D,NaN,1,2,1,0.0
21,N1,1000030,bridge,NAYABARI BOX CULVERT,23.691083,90.545014,11,B,LRP012a,II,D,NaN,1,2,2,0.0
22,N1,1000035,bridge,Madanpur Bridge.(L) (3 bridges at crossing),23.685500,90.551278,28,A,LRP013a,II,D,NaN,1,2,1,0.0


In [11]:
# calculation of vulnerability index for each bridge
df["Vulnerability index"] = (df["Earthquake risk eq"]/3) + (df['Flood risk eq']/4) + (df['Erosion risk eq']/1) + (df["Condition eq"]/4)

print(f"The minimum vulnerability index of a bridge in the dataset is {df['Vulnerability index'].min()}")
print(f"The maximum vulnerability index of a bridge in the dataset is {df['Vulnerability index'].max()}")

The minimum vulnerability index of a bridge in the dataset is 0.8333333333333333
The maximum vulnerability index of a bridge in the dataset is 3.25


In [13]:
df.to_csv("../data/vulnerability_index.csv", encoding="utf-8")